In [ ]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os

print("Running BZ CBBEL Web Scraping Tool v.1.0")

now=datetime.datetime.now()
filename= 'BZ CBBEL data {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

scriptfolder=os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

regdict={'BZ CBBEL 1': 'https://www.centralbank.org.bz/home/core-functions/prudential-supervision/domestic-banks',
'BZ CBBEL 2': 'https://www.centralbank.org.bz/home/core-functions/prudential-supervision/international-banks',
'BZ CBBEL 3': 'https://www.centralbank.org.bz/home/core-functions/prudential-supervision/other-financial-institutions',
'BZ CBBEL 4': 'https://www.centralbank.org.bz/home/core-functions/prudential-supervision/credit-unions',
}

os.chdir(scriptfolder)
#print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))
chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
		"plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}
try:
	os.mkdir(tempfolder)
except:
	prevfiles=os.listdir(tempfolder)
	os.chdir(tempfolder)
	for prf in prevfiles:
		os.remove(prf)
	print('The directory tempfolder already exists.')
os.chdir(tempfolder)##only if files are going to be downloaded here

processdate=now.strftime('%Y-%m-%d')

for reg in regdict:
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    soup=BeautifulSoup(driver.page_source, 'html.parser')
    div=soup.find('div', {'id': 'content_mainContent_C003_ContentDiv'})
    strdiv=str(div).replace('<strong>','<strong>////###').replace('<br>','***').replace('<br/>','***')
    div=BeautifulSoup(strdiv, 'html.parser')
    entities=div.text.split('////')
    entities=list(filter(lambda x:'###' in x and len(x.split('***'))>1,entities))
    for entity in entities:
        rows=entity.split('***')
        print(rows)	
        if len(rows)>2:
            if len(rows[0])>3:    
                sqldict['Name'].append(rows[0].replace("###","").strip())
            else:
                sqldict['Name'].append(rows[1].strip())
            if len(rows[0])>3:  
                sqldict['Address_1'].append(rows[1].strip())
            else:
                sqldict['Address_1'].append(rows[2].strip())
            for row in rows[2:]:
                if 'Telephone:' in row: 
                    sqldict['Phone'].append(row.split(':',1)[1].strip())  
                else: ''
            for row in rows[2:]:
                if 'Website:' in row: 
                    sqldict['Website'].append(row.split(':',1)[1].strip())  
                else:''
            for row in rows[2:]:
                if 'Email:' in row: 
                    sqldict['Email'].append(row.split(':',1)[1].strip())  
                else:''
            sqldict['ListProcessDate'].append(processdate)		
            sqldict['RegCtry'].append('BZ')		
            sqldict['Cntry'].append('BZ')		
            sqldict['RegCode'].append('CBBEL')		
            sqldict['ListCode'].append(reg.split(' ')[-1])		
            sqldict['RegulationType'].append('Licensed')	
            for key in sqldict.keys():			
                while len(sqldict[key])<len(sqldict['ListProcessDate']):				
                    sqldict[key].append('')

		

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()
    
    
    